In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(force=True)

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ.setdefault('USER_AGENT', 'SimpleGenAIApp/1.0')
# LangSmith Tracking
os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGCHAIN_API_KEY')
os.environ['LANGCHAIN_TRACING_V2'] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

In [14]:
# Data Ingestion 

from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://docs.langchain.com/langsmith/home")
docs = loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/home', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='LangSmith docs - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyProfile configurationIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIAudit logsToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageDocumentation IndexFetch the complete documentation index at: https://docs.langchain.com/llms.txtUse this file to discover all a

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitters = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitters.split_documents(docs)
splits

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/home', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='LangSmith docs - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyProfile configurationIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIAudit logsToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageDocumentation IndexFetch the complete documentation index at: https://docs.langchain.com/llms.txtUse this file to discover all a

In [4]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()


In [12]:
from langchain_community.vectorstores import FAISS
db = FAISS.from_documents(splits, embeddings)
db

In [ ]:
### Query from vector store db

db.similarity_search("How to get started with LangSmith?")[0].page_content

'\u200bGet started\nCreate an accountSign up at smith.langchain.com (no credit card required).\nYou can log in with Google, GitHub, or email.Create an API keyGo to your Settings page → API Keys → Create API Key.\nCopy the key and save it securely.Choose your integrationLangSmith works with many frameworks and providers. Browse available integrations to connect your stack.\nOnce your account and API key are ready, choose a quickstart to begin building with LangSmith:\nObservabilityGain visibility into every step your application takes to debug faster and improve reliability.Start tracingEvaluationMeasure and track quality over time to ensure your AI applications are consistent and trustworthy.Evaluate your appDeploymentDeploy your agents as Agent Servers, ready to scale in production.Deploy your agents\n\u200bMore ways to build'

In [15]:
query = "Langsmith has two usage limits: total traces and extended"
result = db.similarity_search(query)
result[0].page_content

'this file to discover all available pages before exploring further.LangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications. Trace requests, evaluate outputs, test prompts, and manage deployments all in one place, with your agent stack.'

In [16]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model = "gpt-4o")

In [21]:
### Retreival Chain, Document Chain 

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
Answer the following question based on the provided context: 
<context>
{context}
</context>

Question: {input}
"""
)

document_chain = create_stuff_documents_chain(llm, prompt)
document_chain


RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the following question based on the provided context: \n<context>\n{context}\n</context>\n\nQuestion: {input}\n'), additional_kwargs={})])
| ChatOpenAI(profile={'name': 'GPT-4o', 'release_date': '2024-05-13', 'last_updated': '2024-08-06', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True

In [24]:
from langchain_core.documents import Document 
document_chain.invoke({
   "input":"How to get started with LangSmith?",
   "context":[Document(page_content="How to get started with LangSmith? Sign up at smith.langchain.com (no credit card required). You can log in with Google, GitHub, or email.")]
     })

'To get started with LangSmith, you need to sign up at smith.langchain.com. You can log in using your Google, GitHub, or email account, and no credit card is required for the sign-up process.'

However, we want the documents to first coome from the reteriver we just set up. That way, we cam use the reteriver to dynamically select the most relevant documents and pass those in for a given question.

In [34]:
## Input --> Retreiver --> VectorStoreDB

reteriver = db.as_retriever()
from langchain_classic.chains import create_retrieval_chain as create_retriever_chain
retreival_chain = create_retriever_chain(reteriver, document_chain)

In [36]:
retreival_chain 

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x11ec0f9d0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the following question based on the provided context: \n<context>\n{context}\n</context>\n\nQuestion: {input}\n'), additional_kwa

In [37]:
## Get the response from the LLM
response = retreival_chain.invoke({"input":"How to get started with LangSmith? Sign up at smith.langchain.com (no credit card required). You can log in with Google, GitHub, or email."})
response['answer']

'To get started with LangSmith, you need to create an account by signing up at smith.langchain.com. This process does not require a credit card. You can log in using Google, GitHub, or email.'

In [38]:
response

{'input': 'How to get started with LangSmith? Sign up at smith.langchain.com (no credit card required). You can log in with Google, GitHub, or email.',
 'context': [Document(id='59e906a8-da99-447b-b336-e45e2b96d4e8', metadata={'source': 'https://docs.langchain.com/langsmith/home', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='\u200bGet started\nCreate an accountSign up at smith.langchain.com (no credit card required).\nYou can log in with Google, GitHub, or email.Create an API keyGo to your Settings page → API Keys → Create API Key.\nCopy the key and save it securely.Choose your integrationLangSmith works with many frameworks and providers. Browse available integrations to connect your stack.\nOnce your account and API key are ready, choose a quickstart to begin building with LangSmith:\nObservabilityGain visibility into every step your application takes to debug faster and improve reliability.Start tracingEvaluationMeasure and track quality over time 

In [39]:
response['context']

[Document(id='59e906a8-da99-447b-b336-e45e2b96d4e8', metadata={'source': 'https://docs.langchain.com/langsmith/home', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='\u200bGet started\nCreate an accountSign up at smith.langchain.com (no credit card required).\nYou can log in with Google, GitHub, or email.Create an API keyGo to your Settings page → API Keys → Create API Key.\nCopy the key and save it securely.Choose your integrationLangSmith works with many frameworks and providers. Browse available integrations to connect your stack.\nOnce your account and API key are ready, choose a quickstart to begin building with LangSmith:\nObservabilityGain visibility into every step your application takes to debug faster and improve reliability.Start tracingEvaluationMeasure and track quality over time to ensure your AI applications are consistent and trustworthy.Evaluate your appDeploymentDeploy your agents as Agent Servers, ready to scale in production.Deploy yo